> **Prefer Google Colab?** [Open this notebook in Colab](https://colab.research.google.com/github/SpatialTurn/DataCollection/blob/main/notebooks/01a_Census_Boundaries.ipynb).
> In Colab, click **File > Save a copy in Drive** first so your changes are kept.

# Census Boundaries: Reading TIGER Files

**Series:** Census Data in Python (notebook 1 of 3)
**Time:** about 30 minutes

**In this notebook you will:**
- Read U.S. Census Bureau boundary files straight from the web into GeoPandas
- Understand FIPS codes and GEOIDs, the IDs that link maps to census tables
- Compare detailed TIGER/Line files with simplified cartographic boundary files
- Filter states, counties, census tracts and roads by their attributes

**You should already know:** how to run a notebook cell and what a pandas DataFrame is. No Census API key is needed for this notebook.

> **Before you start:** run each cell in order by clicking it and pressing **Shift + Enter**.

## Background: Census geography

The Census Bureau publishes the boundaries of every geographic unit it reports on: states, counties,
census tracts, block groups, places (cities and towns) and many more. These come in two families:

| Product | What it is | Best for |
|---|---|---|
| **TIGER/Line** files (`tl_...`) | Full-detail boundaries, including water areas. Shared with roads, rivers and railroads. | Analysis that needs exact boundaries, and features such as roads |
| **Cartographic boundary** files (`cb_...`) | Simplified boundaries clipped to the shoreline, at 1:500,000, 1:5,000,000 or 1:20,000,000 scale | Thematic maps; much smaller and faster to load |

Every unit has a **GEOID** made of **FIPS codes** (Federal Information Processing Standards codes)
joined together. The GEOID is how a census table finds its shape on a map:

| Unit | GEOID pattern | Example | Meaning |
|---|---|---|---|
| State | `SS` | `18` | Indiana |
| County | `SS` + `CCC` | `18157` | Tippecanoe County, Indiana |
| Census tract | `SS` + `CCC` + `TTTTTT` | `18157000100` | Tract 1 in Tippecanoe County |

GEOIDs are **text, not numbers**. Stored as a number, `01` (Alabama) becomes `1` and no longer matches.

## Step 1: Import libraries

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

## Step 2: Read the state boundaries from the Census website

`gpd.read_file` can open a zipped shapefile directly from a web address. The data goes straight into
memory; nothing is saved to your computer. We start with the small 1:20,000,000 cartographic file.

In [ ]:
states = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_state_20m.zip")
print(len(states), "rows")
states.head()

There are 52 rows: the 50 states, the District of Columbia and Puerto Rico. Useful columns:

| Column | Meaning |
|---|---|
| `STATEFP` | State FIPS code |
| `STUSPS` | Two-letter postal abbreviation |
| `NAME` | Name |
| `GEOID` | Geographic identifier (for a state, the same as `STATEFP`) |
| `ALAND`, `AWATER` | Land and water area in square meters |

## Step 3: Find Indiana's FIPS code

Square brackets filter a table: `states[condition]` keeps only the rows where the condition is true.

In [ ]:
indiana = states[states["STUSPS"] == "IN"]
indiana[["NAME", "STUSPS", "STATEFP", "GEOID"]]

Indiana's code is `18`. Let us map the lower 48 states and highlight it.

In [ ]:
lower48 = states[~states["STUSPS"].isin(["AK", "HI", "PR"])]

ax = lower48.plot(color="lightgray", edgecolor="white", figsize=(10, 6))
indiana.plot(ax=ax, color="tab:orange")
ax.set_title("Indiana (FIPS 18) among the lower 48 states")
ax.set_axis_off()
plt.show()

## Step 4: TIGER/Line versus cartographic boundaries

The full TIGER/Line state file is about 50 times larger, because it keeps every detail and includes
water. For Michigan, which borders four of the Great Lakes, the difference is striking.

In [ ]:
tiger_states = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
tiger_states[tiger_states["STUSPS"] == "MI"].plot(ax=axes[0], color="tab:blue")
states[states["STUSPS"] == "MI"].plot(ax=axes[1], color="tab:blue")
axes[0].set_title("TIGER/Line: legal boundary, includes lake area")
axes[1].set_title("Cartographic boundary: clipped to the shoreline")
for ax in axes:
    ax.set_axis_off()
plt.show()

Michigan's legal boundary extends into the Great Lakes, so the TIGER/Line shape includes a lot of
water. For a thematic map of people or places, the cartographic file is the better choice. For
analysis that needs exact legal boundaries, use TIGER/Line.

## Step 5: Counties in Indiana

The county file covers the whole country, so we filter it to Indiana using the code we just found.

In [ ]:
counties = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_county_5m.zip")
in_counties = counties[counties["STATEFP"] == "18"]
print(len(in_counties), "counties in Indiana")
in_counties[["NAME", "STATEFP", "COUNTYFP", "GEOID"]].head()

Indiana has 92 counties. Notice that `GEOID` is simply `STATEFP` followed by `COUNTYFP`.

You can combine conditions with `&` (and). Each condition goes inside its own parentheses:

In [ ]:
tippecanoe = counties[(counties["STATEFP"] == "18") & (counties["NAME"] == "Tippecanoe")]
tippecanoe[["NAME", "COUNTYFP", "GEOID"]]

In [ ]:
ax = in_counties.plot(color="lightgray", edgecolor="white", figsize=(5, 7))
tippecanoe.plot(ax=ax, color="tab:orange")
ax.set_title("Tippecanoe County (GEOID 18157), home of Purdue University")
ax.set_axis_off()
plt.show()

## Try it yourself

Find the county code (`COUNTYFP`) and `GEOID` of **Hamilton County, Indiana**.

Be careful: several states have a Hamilton County, so you need to filter on the state too.

*Hint:* Start from `counties` (the whole country) and combine two conditions with `&`.

<details>
<summary><b>Show a solution</b></summary>

```python
counties[(counties["STATEFP"] == "18") & (counties["NAME"] == "Hamilton")][["NAME", "COUNTYFP", "GEOID"]]
# COUNTYFP is 057 and GEOID is 18057
```

</details>

In [ ]:
# Write your answer here

## Step 6: Roads in Tippecanoe County

TIGER/Line also includes **features** such as roads. Road files are published one county at a time,
named with the county's GEOID: `tl_2024_18157_roads.zip`.

In [ ]:
roads = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_18157_roads.zip")
print(len(roads), "road segments")
roads.head()

The `RTTYP` column is the **route type**. The codes are:

| Code | Route type |
|---|---|
| `I` | Interstate |
| `U` | U.S. highway |
| `S` | State route |
| `C` | County road |
| `M` | Common name (most city streets) |
| `O` | Other |

`value_counts()` counts how many rows have each value:

In [ ]:
roads["RTTYP"].value_counts(dropna=False)

Most segments are local streets (`M`). `None` (shown as `NaN` in newer versions of pandas) means the segment
has no route type, often a driveway, parking lot road or unnamed track. Let us map the major roads on top of the county outline.

In [ ]:
major = roads[roads["RTTYP"].isin(["I", "U", "S"])]

ax = tippecanoe.plot(color="whitesmoke", edgecolor="gray", figsize=(7, 7))
roads.plot(ax=ax, color="lightgray", linewidth=0.3)
major.plot(ax=ax, column="RTTYP", linewidth=2, legend=True,
           categorical=True, cmap="Set1",
           legend_kwds={"title": "Route type", "loc": "lower left"})
ax.set_title("Major roads in Tippecanoe County")
ax.set_axis_off()
plt.show()

For an interactive map, `.explore()` draws the layer on a web map you can pan and zoom. Hover over a
road to see its name. (We use an Esri basemap, which loads in every notebook service, including Google Colab.)

In [ ]:
major.explore(column="RTTYP", tooltip=["FULLNAME", "RTTYP"], categorical=True,
              cmap="Set1", tiles="Esri.WorldGrayCanvas", style_kwds={"weight": 4})

## Try it yourself

Make a map of only the **Interstate** segments (`RTTYP` equal to `"I"`) on top of the county outline.
Which interstate crosses Tippecanoe County? Look at the `FULLNAME` column.

<details>
<summary><b>Show a solution</b></summary>

```python
interstates = roads[roads["RTTYP"] == "I"]
print(list(interstates["FULLNAME"].unique()))     # I- 65
ax = tippecanoe.plot(color="whitesmoke", edgecolor="gray", figsize=(6, 6))
interstates.plot(ax=ax, color="tab:red", linewidth=2)
ax.set_axis_off()
plt.show()
```

</details>

In [ ]:
# Write your answer here

## Step 7: Census tracts

**Census tracts** are small, fairly permanent areas of roughly 1,200 to 8,000 people, drawn so that
neighborhoods can be compared over time. They are the most common unit for neighborhood research.
Tract files are published one state at a time.

In [ ]:
in_tracts = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_18_tract_500k.zip")
tipp_tracts = in_tracts[in_tracts["COUNTYFP"] == "157"]
print(len(in_tracts), "tracts in Indiana;", len(tipp_tracts), "in Tippecanoe County")
tipp_tracts[["NAMELSAD", "STATEFP", "COUNTYFP", "TRACTCE", "GEOID"]].head()

In [ ]:
ax = tipp_tracts.plot(color="lightsteelblue", edgecolor="white", figsize=(7, 7))
ax.set_title("Census tracts in Tippecanoe County")
ax.set_axis_off()
plt.show()

Tracts near the cities of Lafayette and West Lafayette are small because many people live there;
rural tracts are large. The 11-character `GEOID` (state + county + tract) is what we will use in the
next two notebooks to attach census statistics to these shapes.

## Recap

- Census boundary files can be read straight from www2.census.gov with `gpd.read_file`
- Cartographic boundary files (`cb_`) are small and clipped to the shoreline; TIGER/Line files (`tl_`) are complete and include features such as roads
- A GEOID is built from FIPS codes: state (2), county (3) and tract (6) characters
- GEOIDs must stay text so that leading zeros are kept

**Next:** [Getting Data from the Census API](01b_Census_API_Basics.ipynb)